# Hypothesis Testing For Decision Making

Official MA1001B alignment: 6.1 elements; 6.2 intervals and tests; 6.3 p-values; 6.4-6.7 tests for means, proportions, and variances.


## How To Use This Lesson

Read the explanation cells before running the code. Run each code cell in order. When a checkpoint appears, stop and write your answer before continuing. The goal is not only to obtain output; the goal is to justify a decision from data.


## Learning Goals

- Explain the statistical idea in words.
- Implement the idea in Python with readable code.
- Interpret the result as evidence for a decision.
- State at least one assumption or limitation.


## Decision Scenario

A product owner wants to know whether a treatment page should replace a control page. The analysis must separate statistical evidence from practical business value.


## Conceptual Explanation

A hypothesis test asks whether the observed data would be surprising if a null claim were true. The p-value is not the probability that the null is true. It is a probability of data at least as extreme as what was observed, calculated under the null model.


## Mathematical Anchor

For two proportions, the practical effect is p_treatment - p_control. A test can evaluate evidence against equal conversion rates, but the decision should also consider effect size.


## Data And Workflow Notes

Uses a simulated A/B test unless Kaggle A/B testing files are later connected.


## Python Setup


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

rng = np.random.default_rng(1001)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 20)


## Worked Example


In [ ]:
ab = pd.DataFrame({
    "group": np.repeat(["control", "treatment"], 1000),
    "converted": np.r_[rng.binomial(1, 0.10, 1000), rng.binomial(1, 0.125, 1000)],
})
conversion = ab.groupby("group")["converted"].mean()
conversion


In [ ]:
table = pd.crosstab(ab["group"], ab["converted"])
chi2, p_value, dof, expected = stats.chi2_contingency(table)

effect = conversion.loc["treatment"] - conversion.loc["control"]
pd.Series({
    "control_conversion": conversion.loc["control"],
    "treatment_conversion": conversion.loc["treatment"],
    "absolute_lift": effect,
    "p_value": p_value,
}).round(4)


## From Calculation To Evidence


In [ ]:
control = ab.loc[ab["group"].eq("control"), "converted"]
treatment = ab.loc[ab["group"].eq("treatment"), "converted"]

bootstrap_lifts = []
for seed in range(1000):
    c = control.sample(len(control), replace=True, random_state=seed).mean()
    t = treatment.sample(len(treatment), replace=True, random_state=seed + 10_000).mean()
    bootstrap_lifts.append(t - c)

pd.Series(bootstrap_lifts).quantile([0.025, 0.5, 0.975]).round(4)


In [ ]:
minimum_practical_lift = 0.02
pd.Series({
    "observed_lift": effect,
    "meets_practical_threshold": effect >= minimum_practical_lift,
    "statistically_detectable_at_0.05": p_value < 0.05,
})


## Guided Checkpoint

Would you launch the treatment if the p-value is below 0.05 but the lift is smaller than the practical threshold?


## Common Mistakes

- Interpreting p < 0.05 as proof that the treatment is important.
- Ignoring effect size.
- Changing the hypothesis after seeing the result without saying so.


## Independent Practice

Change the treatment conversion rate in the simulation. Find a case with a small p-value but weak practical value, or strong practical value but high uncertainty.


## Interpretation Template

Use this structure for your written answer:

1. The decision question is ...
2. The statistical evidence is ...
3. The uncertainty or limitation is ...
4. Therefore, I recommend ... because ...


## Exit Ticket

What is one sentence you should never write about a p-value?
